<a href="https://colab.research.google.com/github/OJB-Quantum/Notebooks-for-Ideas/blob/main/Gemma_4_in_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Authored by Onri Jay Benally (2026)

Open Access (CC-BY-4.0)

In [1]:
!uv pip install --system PyMuPDF ollama

Using Python 3.12.13 environment at: /usr
Resolved 13 packages in 262ms
Prepared 2 packages in 248ms
Installed 2 packages in 5ms
 + ollama==0.6.1
 + pymupdf==1.27.2.2


In [3]:
"""
Optimized local execution pipeline for the Gemma 4 Large Language Model.

This script seamlessly installs fundamental system dependencies and initializes
the Ollama backend server within a high-capacity Colaboratory environment.
It massively expands the context window to fully utilize 96 Gigabytes of
Video Random Access Memory, subsequently processing user provided files
(Portable Document Format, Text, Markdown, Interactive Python Notebook,
or Python files) or direct text input for local inference.
"""

import os
import json
import time
import socket
import subprocess
import urllib.error
import urllib.request
from pathlib import Path
from dataclasses import dataclass

import fitz  # PyMuPDF library for Portable Document Format extraction
import ollama

# =============================================================================
# Control Knobs
# =============================================================================
@dataclass(frozen=True)
class LocalDeploymentConfig:
    """Configuration parameters for the maximally expanded Ollama deployment."""

    # The precisely corrected identifier for the 31 Billion parameter model.
    model_name: str = "gemma4:31b"

    # The massively expanded context window (128k tokens) for 96GB hardware.
    context_window_size: int = 131072

    # Network configuration for the local daemon process.
    ollama_host: str = "127.0.0.1"
    ollama_port: int = 11434

    # File paths for storing model weights and execution logs.
    models_directory: Path = Path("/content/ollama_models")
    log_file_path: Path = Path("/content/ollama_serve.log")

CONFIGURATION = LocalDeploymentConfig()
# =============================================================================

def execute_bash_command(command: str) -> None:
    """
    Execute a systemic bash command to facilitate environmental setup.

    Args:
        command (str): The specific terminal string to execute.
    """
    print(f"System Command: {command}")
    subprocess.run(["bash", "-lc", command], check=True)

def check_tcp_port_availability(host: str, port: int) -> bool:
    """
    Determine if a local network port is actively accepting connections.

    Args:
        host (str): The local internet protocol address.
        port (int): The specific networking port to inspect.

    Returns:
        bool: An affirmative boolean if the port is open and accessible.
    """
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as network_socket:
        network_socket.settimeout(0.5)
        return network_socket.connect_ex((host, port)) == 0

def await_server_initialization(host: str, port: int, timeout: float = 30.0) -> None:
    """
    Pause execution until the Ollama server responds successfully.

    Args:
        host (str): The local internet protocol address.
        port (int): The specified network port.
        timeout (float): The maximum duration to wait in seconds.
    """
    endpoint_url = f"http://{host}:{port}/api/version"
    deadline_time = time.time() + timeout

    print("Waiting for the Ollama backend to initialize successfully...")
    while time.time() < deadline_time:
        try:
            with urllib.request.urlopen(endpoint_url, timeout=2.0) as response:
                if response.status == 200:
                    print("The Ollama backend is fully operational.")
                    return
        except (urllib.error.URLError, ConnectionResetError):
            time.sleep(1.0)

    raise TimeoutError("The Ollama server failed to initialize within the timeout period.")

def setup_and_launch_ollama() -> None:
    """
    Install the required system tools, download the Ollama binary,
    and launch the required background daemon.
    """
    if not os.path.exists("/usr/local/bin/ollama"):
        print("Installing fundamental hardware diagnostic tools...")
        execute_bash_command("apt-get update -y")
        execute_bash_command("apt-get install -y curl ca-certificates zstd pciutils lshw")

        print("Installing the Ollama application binary...")
        execute_bash_command("curl -fsSL https://ollama.com/install.sh | sh")

    CONFIGURATION.models_directory.mkdir(parents=True, exist_ok=True)

    if check_tcp_port_availability(CONFIGURATION.ollama_host, CONFIGURATION.ollama_port):
        print("The Ollama server is already running continuously in the background.")
        return

    environmental_variables = os.environ.copy()
    environmental_variables["OLLAMA_HOST"] = f"{CONFIGURATION.ollama_host}:{CONFIGURATION.ollama_port}"
    environmental_variables["OLLAMA_MODELS"] = str(CONFIGURATION.models_directory)

    print(f"Starting the background process (logging to {CONFIGURATION.log_file_path})...")
    with open(CONFIGURATION.log_file_path, "a", encoding="utf-8") as log_file:
        subprocess.Popen(
            ["ollama", "serve"],
            env=environmental_variables,
            stdout=log_file,
            stderr=subprocess.STDOUT,
        )

    await_server_initialization(CONFIGURATION.ollama_host, CONFIGURATION.ollama_port)

    print(f"Downloading the massive model weights for {CONFIGURATION.model_name}...")
    execute_bash_command(f"ollama pull {CONFIGURATION.model_name}")

def extract_textual_data(file_path: str) -> str:
    """
    Extract the raw text from a variety of supported file formats.

    Args:
        file_path (str): The precise location of the target file.

    Returns:
        str: The extracted textual contents.
    """
    _, extension = os.path.splitext(file_path)
    extension = extension.lower()

    if extension == '.pdf':
        pdf_document = fitz.open(file_path)
        extracted_content = ""
        for page in pdf_document:
            extracted_content += page.get_text()
        return extracted_content

    if extension == '.ipynb':
        with open(file_path, 'r', encoding='utf8') as notebook:
            notebook_json = json.load(notebook)
            extracted_content = ""
            for cell in notebook_json.get('cells', []):
                if cell.get('cell_type') in ('markdown', 'code'):
                    extracted_content += "".join(cell.get('source', [])) + "\n"
            return extracted_content

    if extension in ('.txt', '.md', '.py'):
        with open(file_path, 'r', encoding='utf8') as standard_text_file:
            return standard_text_file.read()

    raise ValueError(f"The supplied file extension ({extension}) is currently unsupported.")

def execute_inference_cycle() -> None:
    """
    Prompt the user for input and generate a response using the local model.
    """
    print("\nPlease provide a file path or paste your direct text input:")
    user_input = input("Input > ").strip()

    if os.path.isfile(user_input):
        print(f"Processing the localized file: {user_input}")
        document_text = extract_textual_data(user_input)
    else:
        print("Processing the input as a direct textual string.")
        document_text = user_input

    analytical_prompt = f"Please thoroughly analyze the following content:\n\n{document_text}"

    print(f"Generating the model response using a {CONFIGURATION.context_window_size} token context window...")
    model_response = ollama.chat(
        model=CONFIGURATION.model_name,
        messages=[{"role": "user", "content": analytical_prompt}],
        options={"num_ctx": CONFIGURATION.context_window_size}
    )

    print("\nModel Output:\n")
    print(model_response['message']['content'])

if __name__ == '__main__':
    setup_and_launch_ollama()
    execute_inference_cycle()

Starting the background process (logging to /content/ollama_serve.log)...
Waiting for the Ollama backend to initialize successfully...
The Ollama backend is fully operational.
System Command: ollama pull gemma4:31b

Please provide a file path or paste your direct text input:
Input > Can Gemma 4 help me generate a CuPy script for Google Colab that is PEP 8/257 compliant and performs a direct numerical simulation of a standard CFD problem and plots the results at different reasonable time steps in 2D? If so, please proceed. 
Processing the input as a direct textual string.
Generating the model response using a 131072 token context window...

Model Output:

### Analysis of Request

The user is asking for a high-performance computational physics script with specific software engineering and mathematical constraints. 

**Key Requirements:**
1.  **Technology Stack:** Python, **CuPy** (for NVIDIA GPU acceleration), and **Google Colab** compatibility.
2.  **Compliance:** **PEP 8** (style guide